# 📊 01. Khám Phá Dữ Liệu & Vi Cấu Trúc Rổ Cổ Phiếu VN30 (2020 - 2026)
### *Hệ Thống Giao Dịch Định Lượng Cổ Phiếu VN30 & Phái Sinh VN30F1M*

Notebook này cung cấp quy trình phân tích thăm dò dữ liệu (Exploratory Data Analysis - EDA) trên rổ **30 cổ phiếu VN30**, chỉ số **VN30 Index**, **VN-INDEX** và hợp đồng tương lai **VN30F1M** từ 2020 đến 2026:
1. **Thiết lập môi trường và cấu hình đường dẫn dữ liệu.**
2. **Khảo sát ma trận giá đóng cửa và thanh khoản (Volume).**
3. **Phân tích thống kê mô tả (Kurtosis, Skewness, Phân phối lợi suất Fat-Tails).**
4. **Vi cấu trúc thị trường: Chênh lệch Basis ($VN30F1M - VN30$) và tác động trượt giá (5% ADV20).**
5. **Trực quan hóa ma trận tương quan (Correlation Heatmap).**


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn thư mục gốc dự án
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Dir: {DATA_DIR} (Tồn tại: {DATA_DIR.exists()})")

# Cấu hình phong cách biểu đồ trực quan
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120


## 1. Nạp Ma Trận Giá Đóng Cửa & Khối Lượng 30 Cổ Phiếu VN30
Dữ liệu đã được tiền xử lý và chuẩn hóa đồng bộ ngày giao dịch (Trading Calendar alignment).


In [ ]:
# Đọc ma trận giá đóng cửa và khối lượng
close_df = pd.read_csv(DATA_DIR / "1_market_indices" / "VN30_All_Stocks_Close_Prices.csv", parse_dates=['date'])
close_df.set_index('date', inplace=True)

volume_df = pd.read_csv(DATA_DIR / "1_market_indices" / "VN30_All_Stocks_Volumes.csv", parse_dates=['date'])
volume_df.set_index('date', inplace=True)

print(f"Số phiên giao dịch: {len(close_df)}")
print(f"Khoảng thời gian: {close_df.index.min().strftime('%Y-%m-%d')} đến {close_df.index.max().strftime('%Y-%m-%d')}")
print(f"Số lượng cổ phiếu trong rổ: {close_df.shape[1]}")
close_df.tail()


## 2. Thống Kê Mô Tả Lợi Suất & Rủi Ro Đuôi (Fat-Tails)
Tính toán lợi suất logarit hàng ngày $r_t = \ln(P_t / P_{t-1})$ để kiểm định tính phi chuẩn (Non-normality) của thị trường chứng khoán Việt Nam.


In [ ]:
# Tính lợi suất hàng ngày
daily_returns = close_df.pct_change().dropna()

stats_summary = pd.DataFrame({
    'Lợi Suất TB Năm (%)': daily_returns.mean() * 252 * 100,
    'Độ Biến Động Năm (%)': daily_returns.std() * np.sqrt(252) * 100,
    'Độ Lệch (Skewness)': daily_returns.skew(),
    'Độ Nhọn (Kurtosis)': daily_returns.kurtosis(),
    'Max Giảm 1 Ngày (%)': daily_returns.min() * 100,
    'Max Tăng 1 Ngày (%)': daily_returns.max() * 100
})

# Sắp xếp theo biến động tăng dần
stats_summary.sort_values(by='Độ Biến Động Năm (%)', inplace=True)
print("TOP 5 CỔ PHIẾU BIẾN ĐỘNG THẤP NHẤT (Thích hợp trọng số HRP cao):")
display(stats_summary.head())

print("\nTOP 5 CỔ PHIẾU BIẾN ĐỘNG CAO NHẤT (Cần hạn chế trọng số):")
display(stats_summary.tail())


## 3. Khảo Sát Thanh Khoản Trung Bình & Giới Hạn Trượt Giá 5% ADV20
Để quản lý quy mô vốn quỹ từ 5 - 50 tỷ VNĐ, hệ thống khống chế khối lượng đặt lệnh mỗi phiên không được vượt quá **5% ADV20** (Average Daily Volume 20 phiên).


In [ ]:
# Tính ADV20 (Khối lượng trung bình 20 phiên)
adv20_df = volume_df.rolling(window=20).mean()

# Giá trị giao dịch trung bình ngày gần nhất (Giá * Khối lượng) / 1 tỷ VNĐ
latest_turnover = (close_df.iloc[-1] * adv20_df.iloc[-1]) / 1e9

plt.figure(figsize=(14, 6))
bars = plt.bar(latest_turnover.index, latest_turnover.values, color='#1f77b4', edgecolor='black', alpha=0.85)
plt.title("Giá Trị Giao Dịch Trung Bình Ngày 20 Phiên (ADV20 Value - Tỷ VNĐ)", fontsize=14, fontweight='bold')
plt.ylabel("Tỷ VNĐ / Phiên", fontsize=12)
plt.xticks(rotation=90, fontsize=10)
plt.axhline(y=50, color='red', linestyle='--', label='Ngưỡng thanh khoản cực mạnh (> 50 Tỷ/phiên)')
plt.legend()
plt.tight_layout()
plt.show()


## 4. Vi Cấu Trúc Thị Trường: Phân Tích Chênh Lệch Basis VN30F1M vs VN30
Hiện tượng Basis âm sâu liên tục là tín hiệu cảnh báo phân phối (Distribution Regime) hoặc áp lực hedging mạnh từ các tay to.


In [ ]:
vn30_idx = pd.read_csv(DATA_DIR / "1_market_indices" / "VN30_Index_2020_2026.csv", parse_dates=['date']).set_index('date')
vn30f1m = pd.read_csv(DATA_DIR / "1_market_indices" / "VN30F1M_2020_2026.csv", parse_dates=['date']).set_index('date')

# Đồng bộ chỉ mục ngày
merged_idx = pd.merge(vn30_idx[['close']].rename(columns={'close': 'VN30'}),
                      vn30f1m[['close']].rename(columns={'close': 'VN30F1M'}),
                      left_index=True, right_index=True)

merged_idx['Basis'] = merged_idx['VN30F1M'] - merged_idx['VN30']
merged_idx['Basis_MA5'] = merged_idx['Basis'].rolling(5).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [2.5, 1.5]})

ax1.plot(merged_idx.index, merged_idx['VN30'], label='VN30 Index', color='#1f77b4', lw=1.5)
ax1.plot(merged_idx.index, merged_idx['VN30F1M'], label='Phái Sinh VN30F1M', color='#ff7f0e', lw=1, alpha=0.8)
ax1.set_title("So Sánh Diễn Biến Chỉ Số Cơ Sở VN30 & Hợp Đồng Tương Lai VN30F1M", fontsize=14, fontweight='bold')
ax1.set_ylabel("Điểm", fontsize=12)
ax1.legend(loc='upper left')

# Vẽ đồ thị Basis
colors = np.where(merged_idx['Basis'] >= 0, '#2ca02c', '#d62728')
ax2.bar(merged_idx.index, merged_idx['Basis'], color=colors, alpha=0.5, width=1.5, label='Basis hàng ngày')
ax2.plot(merged_idx.index, merged_idx['Basis_MA5'], color='black', lw=1.2, label='Basis MA5')
ax2.axhline(0, color='gray', lw=1, linestyle='--')
ax2.axhline(-5, color='red', lw=1.2, linestyle=':', label='Ngưỡng rủi ro Basis < -5.0')
ax2.set_ylabel("Basis (Điểm)", fontsize=12)
ax2.set_title("Chênh Lệch Basis Spread (VN30F1M - VN30)", fontsize=12, fontweight='bold')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()


## 5. Ma Trận Tương Quan 30 Cổ Phiếu Rổ VN30
Quan sát sự đồng pha và tách biệt tự nhiên giữa các nhóm ngành trên thị trường.


In [ ]:
corr_matrix = daily_returns.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-0.2, vmax=1.0, center=0.4,
            annot=False, square=True, linewidths=0.5, cbar_kws={"shrink": .8})
plt.title("Ma Trận Tương Quan Pearson Lợi Suất 30 Cổ Phiếu VN30 (2020 - 2026)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
